# ENEM 2024 · Sprint 4 — Modelo de referência validado

**Pergunta do projeto:** quanto o perfil socioeconômico explica a nota da redação.
**Alvo:** `y_nota_ok`, a nota das redações corrigidas sem problema de status (os zeros por fuga ao tema,
em branco etc. ficam de fora, como decidido no Plano).

O que este notebook faz, seguindo o Lab da a15:

1. treina o **modelo de referência** (regressão linear) dentro de um **pipeline** (a11) e o avalia com
   **validação cruzada de 5 partes**, reportando **média ± desvio**;
2. mede o **vazamento** dos target encodings calculados na Sprint 2;
3. lê **coeficientes com unidade** e mostra a **multicolinearidade** entre as features de renda;
4. desenha os **resíduos**, registra a **faixa de treino** e mede o **erro por grupo**.

⚠️ Tudo roda **só no conjunto de treino**. O conjunto de teste fica guardado até a escolha do modelo
final (G2): se olharmos o teste agora, ele deixa de ser uma medida honesta.

## 0 · Setup

No Colab, envie `enem2024_features_ordinal.parquet` e `lista_features.csv` para a pasta
`data_processed/` (ou ajuste `PASTA`). Bibliotecas: `pandas`, `pyarrow`, `scikit-learn ≥ 1.3`
(por causa do `TargetEncoder`) e `matplotlib`, todas já instaladas no Colab.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

print("scikit-learn", sklearn.__version__)

PASTA   = os.environ.get("ENEM_OUT", "data_processed")
ARQ     = os.path.join(PASTA, "enem2024_features_ordinal.parquet")
ALVO    = "y_nota_ok"
SEED    = 42
AMOSTRA = 400_000                               # linhas de treino usadas na CV (None = todas)
CV5     = KFold(5, shuffle=True, random_state=0)  # convenção do curso: 5 partes, embaralhadas, semente fixa

pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})

## 1 · Leitura

Lemos só as colunas necessárias e só as linhas de **treino** com nota válida. A amostra de 400 mil
linhas deixa a CV rápida no Colab; com `AMOSTRA = None` roda na base de treino inteira.

In [ ]:
FEATURES = pd.read_csv(os.path.join(PASTA, "lista_features.csv"))["feature"].tolist()
CAT = ["cor_raca", "regiao", "uf"]                      # categóricas
TE  = ["te_municipio", "te_uf"]                         # target encodings da Sprint 2
NUM = [c for c in FEATURES if c not in CAT]             # numéricas (inclui os TE)
NUM_SEM_TE = [c for c in NUM if c not in TE]

base   = pd.read_parquet(ARQ, columns=["split", ALVO] + FEATURES)
treino = base[(base["split"] == "treino") & base[ALVO].notna()]
df = treino if AMOSTRA is None else treino.sample(min(AMOSTRA, len(treino)), random_state=SEED)
df = df.reset_index(drop=True)
del base, treino

X = df[FEATURES].copy()
# float64: em float32 a regressão linear perde precisão (renda em R$ tem escala grande).
# Foi o que vimos antes da G1: o R² caía de ~0,27 para ~0,08 sem nenhum erro aparente.
X[NUM] = X[NUM].astype("float64")
X[CAT] = X[CAT].astype(str)
y = df[ALVO].astype("float64")

print(f"linhas: {len(X):,} | numéricas: {len(NUM)} | categóricas: {len(CAT)}")
print(f"nota: média {y.mean():.1f} | desvio {y.std():.1f} | mín {y.min():.0f} | máx {y.max():.0f}")

## 2 · Modelo de referência: regressão linear validada

O pipeline junta o pré-processamento e o modelo, para que **tudo** seja ajustado só com as linhas de
treino de cada rodada da CV (a11).

- As numéricas já chegam tratadas da Sprint 2 (imputação + flags), então passam direto.
- `cor_raca` e `uf` viram one-hot **dentro** do pipeline. Deixamos `regiao` de fora no linear: ela é
  uma função da UF, e as duas juntas são colinearidade perfeita.
- Ficam de fora também `te_municipio` e `te_uf`, que **vazam** na CV (seção 3).

O **chute** (sempre prever a média do treino) é o piso: R² = 0 por definição.

In [ ]:
CAT_LIN = ["cor_raca", "uf"]
METRICAS = {"RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error", "R²": "r2"}

def pipeline_linear(numericas, onehot=CAT_LIN, target_enc=()):
    # monta o pipeline: numéricas direto, categóricas em one-hot e, se pedido, target encoding
    # calculado DENTRO de cada rodada (o TargetEncoder do sklearn ainda faz cross-fitting interno)
    partes = [("num", "passthrough", list(numericas))]
    if onehot:
        partes.append(("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), list(onehot)))
    if target_enc:
        partes.append(("te", TargetEncoder(target_type="continuous", random_state=SEED), list(target_enc)))
    return Pipeline([("prep", ColumnTransformer(partes)), ("modelo", LinearRegression())])

def avaliar(nome, modelo, X_=X):
    # roda a CV de 5 partes e devolve média e desvio de cada métrica
    r = cross_validate(modelo, X_, y, cv=CV5, scoring=METRICAS, n_jobs=-1)
    linha = {"modelo": nome}
    for m in METRICAS:
        v = r[f"test_{m}"] * (1 if m == "R²" else -1)    # o sklearn devolve erros com sinal negativo
        linha[m], linha[f"{m} dp"] = v.mean(), v.std()
    return linha

def tabela(linhas):
    t = pd.DataFrame(linhas).set_index("modelo")
    return pd.DataFrame({m: t[m].map("{:.3f}".format) + " ± " + t[f"{m} dp"].map("{:.3f}".format)
                         for m in METRICAS})

baseline = pipeline_linear(NUM_SEM_TE)
res_base = [avaliar("Chute: média do treino", DummyRegressor(strategy="mean")),
            avaliar("Regressão linear (referência)", baseline)]
tabela(res_base)

**Como ler:** cada célula é **média ± desvio das 5 rodadas**. O RMSE e o MAE estão em **pontos de
nota**. Um modelo novo só é "melhor" que a referência se a diferença for maior que essas barras de erro
(a15: "um R² sem o desvio padrão é uma opinião").

> ✍️ **Leitura do grupo** (preencher depois de rodar): o modelo de referência erra, em média, ___ pontos
> (MAE) e explica ___ da variação da nota (R²), contra ___ pontos de erro do chute.

## 3 · O vazamento do target encoding

`te_municipio` e `te_uf` foram calculados na Sprint 2 com **todo** o conjunto de treino. Na CV, isso
quer dizer que a nota de cada linha de validação **ajudou a calcular** o encoding que o modelo usa para
prevê-la: o modelo vê um pedaço da resposta. É o vazamento da a11.

Comparamos três versões, nas **mesmas** 5 rodadas:

1. **com os TE pré-calculados** (vazam);
2. **sem TE** (a referência da seção 2);
3. **TE da UF recalculado dentro do pipeline**, sem vazamento.

O município não dá para recalcular aqui: a base salva não guarda o código do município. Fica como
tarefa da Sprint 5 (regerar a base com `CO_MUNICIPIO_PROVA`).

In [ ]:
res_te = [avaliar("1 · com te_municipio e te_uf da Sprint 2 (vazam)", pipeline_linear(NUM)),
          res_base[1] | {"modelo": "2 · sem TE (referência)"},
          avaliar("3 · TE da UF dentro do pipeline", pipeline_linear(NUM_SEM_TE, onehot=["cor_raca"],
                                                                     target_enc=["uf"]))]
tabela(res_te)

**Como ler:** se a versão 1 parece melhor que a 2 por **mais** que as barras de erro, parte desse ganho
é otimismo do vazamento, e o número não se repetiria em dados novos. A versão 3 mostra quanto da
informação geográfica dá para usar **sem** vazar.

> ✍️ **Leitura do grupo:** a diferença entre 1 e 2 foi de ___ no R². Por isso, a partir daqui os TE
> pré-calculados saem do pipeline, e o TE passa a ser calculado dentro de cada rodada.

## 4 · O que o modelo diz: coeficientes com unidade

Com todas as features, os coeficientes não são legíveis: há colunas que são combinação exata de outras
(`renda_sm` = `renda_familiar` / 1.412, `esc_media_pais` = média de `esc_pai` e `esc_mae`, as flags de
escola somam 1). O modelo acerta igual, mas "divide o crédito" de forma arbitrária.

Para **ler** os pesos, usamos um modelo **enxuto**, com uma coluna por ideia. Primeiro conferimos quanto
de acerto ele perde em relação à referência.

In [ ]:
ENXUTO = ["log_renda_pc", "esc_max_pais", "indice_bens", "escola_privada", "escola_mista",
          "bolsista", "sexo_fem", "idade", "treineiro", "acesso_digital"]

enxuto = pipeline_linear(ENXUTO)
tabela([res_base[1], avaliar("Linear enxuto (10 numéricas + cor/raça + UF)", enxuto)])

In [ ]:
UNIDADE = {
    "log_renda_pc":   "por +1 no log da renda per capita (dobrar a renda = coef × 0,69)",
    "esc_max_pais":   "por nível a mais de escolaridade do pai/mãe mais escolarizado (escala 0–6)",
    "indice_bens":    "por +1 desvio padrão no índice de bens",
    "escola_privada": "privada vs. pública (referência)",
    "escola_mista":   "mista vs. pública (referência)",
    "bolsista":       "bolsista vs. não bolsista",
    "sexo_fem":       "feminino vs. masculino",
    "idade":          "por ano a mais de idade",
    "treineiro":      "treineiro vs. não treineiro",
    "acesso_digital": "tem computador e wi-fi vs. não tem",
}

def coeficientes(pipe, colunas):
    pipe.fit(X, y)
    nomes = pipe.named_steps["prep"].get_feature_names_out()
    coef = pd.Series(pipe.named_steps["modelo"].coef_, index=nomes)
    return coef[[f"num__{c}" for c in colunas]].set_axis(colunas)

coef_enxuto = coeficientes(enxuto, ENXUTO)
pd.DataFrame({"coef (pontos)": coef_enxuto.round(1), "leitura": pd.Series(UNIDADE)})

**Como ler um coeficiente** (a15): é a diferença de nota associada à feature **mantendo todas as outras
constantes**. Por exemplo, `escola_privada` compara alunos de privada e de pública **com a mesma renda,
a mesma escolaridade dos pais, a mesma UF** etc.

⚠️ É **associação, não causa**. O coeficiente não autoriza dizer "mudar o aluno para uma escola privada
aumenta a nota em X pontos": a escola pode estar no lugar de algo que não medimos (o confundidor da a03).

> ✍️ **Leitura do grupo** (escolher dois): mantendo o resto constante, estudar em escola privada está
> associado a ___ pontos a mais que na pública. Dobrar a renda per capita está associado a ___ pontos.

## 5 · Multicolinearidade: quando duas colunas dizem a mesma coisa

`inse_proxy` é a média dos z-scores de `log_renda_pc`, `esc_max_pais` e `indice_bens`. Se ele entra no
modelo enxuto, as quatro colunas passam a **dividir o crédito** (o exemplo da garagem da a15).

In [ ]:
print("correlação do inse_proxy com as três colunas que o compõem:")
print(X[["inse_proxy", "log_renda_pc", "esc_max_pais", "indice_bens"]].corr()["inse_proxy"].round(2)[1:])

coef_mais = coeficientes(pipeline_linear(ENXUTO + ["inse_proxy"]), ENXUTO + ["inse_proxy"])
comparacao = pd.DataFrame({"sem inse_proxy": coef_enxuto, "com inse_proxy": coef_mais}).round(1)
comparacao.loc[["log_renda_pc", "esc_max_pais", "indice_bens", "inse_proxy"]]

**Como ler:** se os coeficientes de renda, escolaridade e bens mudam bastante quando o `inse_proxy`
entra, mas o acerto do modelo quase não muda, é a multicolinearidade: o que fica instável é a
**leitura**, não a previsão. É o argumento para a **regularização** (Ridge, a20), que já está no Plano.

## 6 · Resíduos: o erro é parecido ao longo da nota?

Usamos as previsões **fora da amostra** (`cross_val_predict`): cada linha é prevista pela rodada em
que ela estava na validação.

In [ ]:
previsto = cross_val_predict(baseline, X, y, cv=CV5, n_jobs=-1)
residuo  = y - previsto

print(f"nota real:     {y.min():.0f} a {y.max():.0f}")
print(f"nota prevista: {previsto.min():.0f} a {previsto.max():.0f}")

idx = np.random.default_rng(SEED).choice(len(y), size=min(20_000, len(y)), replace=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(previsto[idx], residuo.iloc[idx], s=3, alpha=.15, color="#06406F")
ax[0].axhline(0, color="#c2410c", lw=1.2)
ax[0].set_xlabel("nota prevista"); ax[0].set_ylabel("resíduo (real − previsto)")
ax[0].set_title("resíduo × previsão")

faixa = pd.cut(y, bins=range(0, 1001, 100), include_lowest=True)
por_faixa = pd.DataFrame({"faixa": faixa, "residuo": residuo}).groupby("faixa", observed=True)["residuo"]
por_faixa.mean().plot(kind="bar", ax=ax[1], color="#06406F")
ax[1].axhline(0, color="#777", lw=1)
ax[1].set_xlabel("faixa da nota REAL"); ax[1].set_ylabel("resíduo médio")
ax[1].set_title("o modelo erra para que lado, em cada faixa")
plt.tight_layout(); plt.show()

pd.DataFrame({"linhas": por_faixa.size(), "resíduo médio": por_faixa.mean().round(1),
              "erro absoluto mediano": por_faixa.apply(lambda r: r.abs().median()).round(1)})

**Como ler** (os três formatos da a15): uma nuvem **sem forma** em volta do zero é o ideal; um
**leque** quer dizer que o erro cresce numa parte da faixa; um **arco** ou uma faixa inclinada quer dizer
que o modelo erra **sistematicamente** para um lado.

Com um R² moderado, o esperado é que o modelo **puxe as notas para a média**: a faixa de notas previstas
fica bem mais estreita que a real, então ele **subestima** quem tirou nota alta (resíduo médio
positivo) e **superestima** quem tirou nota baixa (negativo). Isso é coerente com a pergunta do projeto:
o perfil explica uma parte da nota, e o resto (escrita, preparo, correção) o perfil não vê.

> ✍️ **Leitura do grupo:** a nota prevista vai de ___ a ___, contra ___ a ___ da real. O formato dos
> resíduos é ___, e isso significa que ___.

## 7 · Em que faixa o modelo vale

A a15 pede para dizer **onde** o modelo foi treinado, porque fora disso ele responde "com a mesma cara"
sem avisar (extrapolação).

In [ ]:
FAIXA = ["renda_pc", "idade", "pessoas_casa", "esc_max_pais"]
pd.DataFrame({"mín": X[FAIXA].min(), "p1": X[FAIXA].quantile(.01),
              "p99": X[FAIXA].quantile(.99), "máx": X[FAIXA].max()}).round(1)

**O modelo vale para:** participantes do **ENEM 2024**, com redação **corrigida sem problema de status**
(nota entre ___ e 1.000), com renda, idade e composição familiar **declaradas no questionário** e dentro
das faixas acima. Ele **não** vale para outros anos, para prever quem zera a redação nem para alunos
individuais fora desse perfil.

## 8 · Erro por grupo: o modelo erra igual para todos?

Primeiro olhar de **equidade** (10% da nota da G2). Para cada grupo: o erro típico (MAE, RMSE) e o
**resíduo médio**. Resíduo médio **positivo** quer dizer que o modelo **subestima** a nota daquele grupo;
**negativo**, que **superestima**.

In [ ]:
tipo_escola = np.select([X["escola_privada"] == 1, X["escola_mista"] == 1, X["escola_publica"] == 1],
                        ["privada", "mista", "pública"], default="sem EM / outra")

def erro_por_grupo(grupo):
    aux = pd.DataFrame({"grupo": grupo, "res": residuo.to_numpy()})
    g = aux.groupby("grupo")["res"]
    return pd.DataFrame({"linhas": g.size(),
                         "MAE": g.apply(lambda r: r.abs().mean()),
                         "RMSE": g.apply(lambda r: np.sqrt((r ** 2).mean())),
                         "resíduo médio": g.mean()}).round(1).sort_values("linhas", ascending=False)

display(erro_por_grupo(tipo_escola))
display(erro_por_grupo(X["cor_raca"].to_numpy()))

**Como ler:** um MAE bem maior num grupo quer dizer que o modelo é **menos confiável** para ele; um
resíduo médio longe de zero quer dizer **viés sistemático** (o modelo erra sempre para o mesmo lado
naquele grupo). O Plano definiu como "bom o suficiente" ter erro parecido entre tipo de escola e
cor/raça. `cor_raca` está no modelo **para medir equidade**, não como causa da nota.

> ✍️ **Leitura do grupo:** o maior MAE foi em ___ (___ pontos) e o menor em ___ (___). O grupo com
> resíduo médio mais distante de zero foi ___.

## 9 · Resumo da Sprint 4 e próximos passos

| | |
|---|---|
| **Modelo de referência** | Regressão linear em pipeline, CV de 5 partes: RMSE ___ ± ___, MAE ___ ± ___, R² ___ ± ___ |
| **Chute (média)** | RMSE ___ |
| **Vazamento** | Os TE da Sprint 2 inflavam o R² em ___. Saem do pipeline; TE da UF passa a ser calculado em cada rodada |
| **Coeficientes** | Privada vs. pública: ___ pontos · dobrar a renda per capita: ___ pontos (associação, não causa) |
| **Resíduos** | ___ |
| **Equidade** | ___ |

**Próximos passos (Sprint 5):** regerar a base com o código do município, para o TE municipal entrar
dentro do pipeline; comparar Ridge, Random Forest e LightGBM contra esta referência, nas **mesmas** 5
rodadas; ajustar hiperparâmetros.